In [ ]:
"""
IF Baseline for HUMS2023 Machinery Fault Detection

Overview:
    This script implements a baseline anomaly detection pipeline using the
    IF algorithm on the HUMS2023 dataset (RF2 sensor). It is designed to align with the
    AIRL/AE/OCSVM experimental framework, producing consistent memmap and NPZ
    outputs for downstream thresholding and evaluation.

Functionality:
    • Loads segmented 1D vibration signals from `.mat` files.
    • Trains multiple IF models across random seeds for ensemble analysis.
    • Computes anomaly scores (−log-likelihoods) for both training and test sets.
    • Saves results as memory-mapped `.npy` arrays and compressed `.npz` archives.
    • Logs all metadata and progress for reproducibility.
"""

import os, json, glob, time, random
import numpy as np
from scipy.io import loadmat
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from numpy.lib.format import open_memmap
import torch  # only for device print consistency with AE

DATASET_PAIRS = [
   ("./data/training_until20",
     ".data//testAIRL_from21",
     "HUMS_RF2"),
]

OUTPUT_DIR = "./if_alreadyStndzscores_HUMSRF2_runs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


IF_N_ESTIMATORS  = 400
IF_MAX_SAMPLES   = 256  
IF_MAX_FEATURES  = 1.0
IF_CONTAMINATION = "auto"   
IF_BOOTSTRAP     = False
IF_N_JOBS        = -1

# Multi-run setup
N_RUNS = 10
SEEDS  = list(range(N_RUNS))


EXPECTED_LEN = 4095 # 4096 for IMS and XJTUSY

PREFERRED_MAT_KEY = "xr" 


USE_SCALER = False
REMOVE_DC  = False


stamp = "HUMSRF2"


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)

def load_folder_mat_1d(folder, preferred_key="xr", expected_len=4095):
    files = sorted(glob.glob(os.path.join(folder, "*.mat")))
    X, F = [], []
    for fp in files:
        m = loadmat(fp)
        
        arr = m.get(preferred_key, None)
        if arr is None:
            for v in m.values():
                if isinstance(v, np.ndarray) and v.size == expected_len:
                    arr = v; break
        if arr is None:
            continue
        x = np.asarray(arr, dtype=np.float32).reshape(-1)
        if expected_len and len(x) != expected_len:
            continue
        X.append(x); F.append(fp)
    if not X:
        raise RuntimeError(f"No usable .mat in {folder}")
    X = np.vstack(X)
    X = np.nan_to_num(X, copy=False)  
    return X, np.array(F, dtype=object)

def standardize_train_test(Xtr_raw, Xte_raw):
    Xtr, Xte = Xtr_raw.copy(), Xte_raw.copy()
    if REMOVE_DC:
        Xtr = Xtr - Xtr.mean(axis=1, keepdims=True)
        Xte = Xte - Xte.mean(axis=1, keepdims=True)
    if USE_SCALER:
        sc = StandardScaler().fit(Xtr)
        Xtr = sc.transform(Xtr); Xte = sc.transform(Xte)
        scaler_mean = sc.mean_.astype(np.float32)
        scaler_scale = sc.scale_.astype(np.float32)
    else:
        d = Xtr.shape[1]
        scaler_mean = np.zeros(d, dtype=np.float32)
        scaler_scale = np.ones(d,  dtype=np.float32)
    return Xtr, Xte, scaler_mean, scaler_scale

def fit_iforest(Xtr, seed):
    max_samples = len(Xtr) if IF_MAX_SAMPLES is None else min(IF_MAX_SAMPLES, len(Xtr))
    clf = IsolationForest(
        n_estimators=IF_N_ESTIMATORS,
        max_samples=max_samples,
        max_features=IF_MAX_FEATURES,
        contamination=IF_CONTAMINATION,
        bootstrap=IF_BOOTSTRAP,
        random_state=seed,
        n_jobs=IF_N_JOBS,
    )
    clf.fit(Xtr)
    return clf

def anomaly_scores(clf, X):
    
    return -clf.score_samples(X).astype(np.float32)

def ensure_channel_dir(tag, stamp):
    chan_dir = os.path.join(OUTPUT_DIR, f"{tag}_{stamp}")
    os.makedirs(chan_dir, exist_ok=True)
    return chan_dir

def write_json_safely(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

def open_or_init_memmaps(chan_dir, n_runs, n_train, n_test):
    tr_path = os.path.join(chan_dir, "train_scores_runs_IFHUMSRF2.npy")
    te_path = os.path.join(chan_dir, "test_scores_runs_IFHUMSRF2.npy")
    if os.path.exists(tr_path) and os.path.exists(te_path):
        mm_tr = open_memmap(tr_path, mode="r+")
        mm_te = open_memmap(te_path, mode="r+")
        # resume point = first all-NaN row
        start = 0
        for r in range(mm_tr.shape[0]):
            if np.all(np.isnan(mm_tr[r])):
                start = r; break
        else:
            start = mm_tr.shape[0]
    else:
        mm_tr = open_memmap(tr_path, mode="w+", dtype="float32", shape=(n_runs, n_train))
        mm_te = open_memmap(te_path, mode="w+", dtype="float32", shape=(n_runs, n_test))
        mm_tr[:] = np.nan; mm_te[:] = np.nan
        mm_tr.flush(); mm_te.flush()
        start = 0
    return mm_tr, mm_te, start


manifest = []

for train_dir, test_dir, tag in DATASET_PAIRS:
    print(f"\n=== Channel: {tag} ===")
    Xtr_raw, Ftr = load_folder_mat_1d(train_dir, PREFERRED_MAT_KEY, EXPECTED_LEN)
    Xte_raw, Fte = load_folder_mat_1d(test_dir,  PREFERRED_MAT_KEY, EXPECTED_LEN)
    assert Xtr_raw.shape[1] == EXPECTED_LEN and Xte_raw.shape[1] == EXPECTED_LEN

    Xtr, Xte, scaler_mean, scaler_scale = standardize_train_test(Xtr_raw, Xte_raw)
    chan_dir = ensure_channel_dir(tag, stamp)

    meta = dict(
        tag=tag, stamp=stamp,
        train_dir=train_dir, test_dir=test_dir,
        n_train=int(Xtr.shape[0]), n_test=int(Xte.shape[0]),
        seeds=SEEDS,
        expected_len=EXPECTED_LEN, preferred_mat_key=PREFERRED_MAT_KEY,
        remove_dc=REMOVE_DC, use_scaler=USE_SCALER,
        scaler_mean=scaler_mean.tolist(),
        scaler_scale=scaler_scale.tolist(),
        if_params=dict(
            n_estimators=IF_N_ESTIMATORS,
            max_samples=(len(Xtr) if IF_MAX_SAMPLES is None else min(IF_MAX_SAMPLES, len(Xtr))),
            max_features=IF_MAX_FEATURES,
            contamination=IF_CONTAMINATION,
            bootstrap=IF_BOOTSTRAP,
            n_jobs=IF_N_JOBS,
        ),
        train_files=[str(x) for x in Ftr],
        test_files=[str(x) for x in Fte],
        device=str(DEVICE),
    )
    write_json_safely(os.path.join(chan_dir, "meta.json"), meta)

    mm_tr, mm_te, start_run = open_or_init_memmaps(chan_dir, N_RUNS, Xtr.shape[0], Xte.shape[0])
    print(f"memmaps: {mm_tr.shape} train, {mm_te.shape} test | resume from run {start_run}/{N_RUNS}")

    prog_path = os.path.join(chan_dir, "progress.json")
    if os.path.exists(prog_path):
        prog = json.load(open(prog_path))
        start_run = max(start_run, int(prog.get("last_completed_run", -1)) + 1)

    for r in range(start_run, N_RUNS):
        seed = SEEDS[r]
        set_seed(seed)
        t0 = time.time()
        clf = fit_iforest(Xtr, seed)
        mm_tr[r] = anomaly_scores(clf, Xtr)
        mm_te[r] = anomaly_scores(clf, Xte)
        mm_tr.flush(); mm_te.flush()
        print(f"  [run {r+1}/{N_RUNS} | seed={seed}] wrote row {r}  ({time.time()-t0:.2f}s)")
        write_json_safely(prog_path, dict(last_completed_run=r, seeds=SEEDS))


    final_npz = os.path.join(chan_dir, f"if_raw4095HUMS_runs_{tag}_{stamp}.npz")
    np.savez_compressed(
        final_npz,
        train_scores_runs=np.array(mm_tr),
        test_scores_runs=np.array(mm_te),
        train_files=Ftr,
        test_files=Fte,
        seeds=np.array(SEEDS, dtype=int),
        scaler_mean=scaler_mean,
        scaler_scale=scaler_scale,
        params=np.array(meta["if_params"], dtype=object),
    )
    print(f"[saved] {final_npz}")

    manifest.append(dict(tag=tag, channel_dir=chan_dir, npz=final_npz,
                         n_train=int(Xtr.shape[0]), n_test=int(Xte.shape[0]), seeds=SEEDS))

man_path = os.path.join(OUTPUT_DIR, f"manifest_{stamp}.json")
write_json_safely(man_path, manifest)
print(f"\n[done] Wrote manifest: {man_path}")


In [ ]:
"""
IF Thresholding and Earliest Detection Analysis for HUMS2023
Overview:
    This script applies multiple statistical and adaptive thresholding methods
    to anomaly scores generated by the IF baseline (or other compatible
    methods) for the HUMS2023 dataset. It aggregates multi-run outputs into
    ensemble scores, evaluates various threshold criteria, and identifies the
    earliest detected anomalies based on natural file ordering.

Functionality:
    • Loads per-run anomaly score arrays (.npz) produced by the IF trainer.
    • Aggregates runs using mean, median, or vote-based ensembles.
    • Applies multiple thresholding techniques:
        - Statistical: mean ± kσ, quantiles (p90–p99), median + 3MAD.
        - Adaptive: K-means midpoint, Otsu’s threshold.
        - Extreme-tail: GPD tail fit (95% quantile).
    • Supports natural or filename-based chronological ranking of test files.
    • Outputs CSV reports summarizing anomaly counts and earliest detections.
    • Prints concise console summaries matching the AIRL-style output format.
"""


import os, re, csv, glob, json
import numpy as np
from sklearn.cluster import KMeans
from skimage.filters import threshold_otsu
from scipy.stats import genpareto

# ------------------------ USER SETTINGS ------------------------
# Match the trainer's output naming (4095 + HUMS in filename)
NPZ_PATHS = sorted(glob.glob("./if_alreadyStndzscores_HUMSRF2_runs/*/if_raw4095HUMS_runs_*.npz"))

# Ensemble toggles (enable exactly one)
USE_MEAN   = True
USE_MEDIAN = False
USE_VOTE   = False  # vote = majority across runs using per-run p99 on TRAIN

ZSCORE_PER_RUN_BEFORE_ENSEMBLE = True
LOWER_TAIL = False  # keep 'higher = more anomalous' convention; set True to flip if needed

# Optionally filter TRAIN for calibration (regex over filenames)
CALIBRATION_FILTER_REGEX = None
SEED = 0

OUT_DIR = "./ifHUMS_threshold_results_runs"  # AE-style output root
os.makedirs(OUT_DIR, exist_ok=True)

# >>> NEW <<< EPS for >= comparisons (float tolerance)
EPS = 1e-12


# If filenames lack Day(\d+), choose the fallback for chronology ranking
CHRONOLOGY_FALLBACK = "natural"  # one of {"natural","alpha","mtime"}

# ------------------------ HELPERS ------------------------
def _natural_key(name: str):
    parts = re.split(r"(\d+)", name)
    key = []
    for p in parts:
        if p.isdigit():
            key.append((1, int(p)))
        else:
            key.append((0, p))
    return tuple(key)

def _check_mode():
    modes = [USE_MEAN, USE_MEDIAN, USE_VOTE]
    if sum(bool(x) for x in modes) != 1:
        raise ValueError("Enable exactly one of USE_MEAN / USE_MEDIAN / USE_VOTE")
    if USE_MEAN:   return "mean"
    if USE_MEDIAN: return "median"
    return "vote"

def ensemble_scores(train_runs, test_runs, zscore_per_run=True):
    R = int(train_runs.shape[0])
    tr = train_runs.astype(float).copy()
    te = test_runs.astype(float).copy()

    if zscore_per_run:
        for r in range(R):
            mu = train_runs[r].mean()
            sd = train_runs[r].std() + 1e-12
            tr[r] = (train_runs[r] - mu) / sd
            te[r] = (test_runs[r]  - mu) / sd

    mode_name = _check_mode()
    if mode_name == "mean":
        return tr.mean(axis=0), te.mean(axis=0), mode_name
    if mode_name == "median":
        return np.median(tr, axis=0), np.median(te, axis=0), mode_name

    taus = np.percentile(tr, 99, axis=1)
    votes_tr = (tr > taus[:, None]).astype(int)
    votes_te = (te > taus[:, None]).astype(int)
    tr_score = votes_tr.mean(axis=0)
    te_score = votes_te.mean(axis=0)
    return tr_score, te_score, mode_name

def build_thresholds(train_scores):
    tr = np.asarray(train_scores, float)
    mu    = float(tr.mean())
    sigma = float(tr.std() + 1e-12)
    n     = int(len(tr))
    se    = sigma / max(1, np.sqrt(n))

    thresholds = {}
    thresholds['mean_plus_2std']   = mu + 2 * sigma
    thresholds['mean_plus_3std']   = mu + 3 * sigma
    thresholds['mean_minus_3std']  = mu - 3 * sigma

    thresholds['p90'] = float(np.percentile(tr, 90))
    thresholds['p95'] = float(np.percentile(tr, 95))
    thresholds['p97'] = float(np.percentile(tr, 97))
    thresholds['p98'] = float(np.percentile(tr, 98))
    thresholds['p99'] = float(np.percentile(tr, 99))

    thresholds['max']              = float(tr.max())
    thresholds['max_minus_se']     = float(tr.max() - se)
    thresholds['max_minus_2se']    = float(tr.max() - 2 * se)

    median = float(np.median(tr))
    mad    = float(np.median(np.abs(tr - median)) + 1e-12)
    thresholds['median_plus_3mad'] = median + 3 * mad

    try:
        k2 = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(tr.reshape(-1, 1))
        centers = np.sort(k2.cluster_centers_.flatten())
        thresholds['kmeans_mid'] = float(0.5 * (centers[0] + centers[1]))
    except Exception:
        pass

    try:
        thresholds['otsu'] = float(threshold_otsu(tr))
    except Exception:
        pass

    try:
        tail_n    = max(10, int(0.05 * len(tr)))
        tail_vals = np.sort(tr)[-tail_n:]
        shift     = float(tail_vals.min())
        shape, loc, scale = genpareto.fit(tail_vals - shift, floc=0)
        gpd_q     = genpareto.ppf(0.95, shape, loc, scale)
        thresholds['gpd_tail95'] = float(shift + gpd_q)
    except Exception:
        pass

    return thresholds

def filter_train_for_calibration(train_scores, train_files, regex):
    if not regex:
        return np.asarray(train_scores)
    pat = re.compile(regex)
    keep = [i for i, f in enumerate(train_files) if pat.search(os.path.basename(str(f)) or "")]
    if not keep:
        return np.asarray(train_scores)
    return np.asarray(train_scores)[keep]

# >>> NEW <<< Natural-order indices (no Day() priority; exactly like your AIRL script’s behavior)
def natural_order_indices(file_list):
    names = [os.path.basename(str(f)) for f in file_list]
    return np.array(sorted(range(len(names)), key=lambda i: _natural_key(names[i])), dtype=int)


def parse_day(filename):
    m = re.search(r"Day(\d+)", os.path.basename(str(filename)))
    return int(m.group(1)) if m else None

def chronology_indices(file_list):
    """
    Prefer Day(\\d+) if present; otherwise fall back per CHRONOLOGY_FALLBACK:
      - "natural": numeric-aware filename ordering
      - "alpha":   pure alphabetical
      - "mtime":   file modification time
    Returns:
      order (np.ndarray of indices), days (np.ndarray of ints or NaN)
    """
    days = [parse_day(f) for f in file_list]
    if any(d is not None for d in days):
        sub = [d if d is not None else 10**9 for d in days]
        order = np.argsort(np.array(sub))
        return order, np.array([d if d is not None else np.nan for d in days], dtype=float)

    names = np.array([os.path.basename(str(f)) for f in file_list])

    if CHRONOLOGY_FALLBACK.lower() == "natural":
        return np.array(sorted(range(len(names)), key=lambda i: _natural_key(names[i])), dtype=int), np.full(len(names), np.nan, dtype=float)

    if CHRONOLOGY_FALLBACK.lower() == "mtime":
        mtimes = np.array([os.path.getmtime(str(f)) for f in file_list], dtype=float)
        return np.argsort(mtimes), np.full(len(names), np.nan, dtype=float)

    return np.argsort(names), np.full(len(names), np.nan, dtype=float)

# ------------------------ MAIN ------------------------
for npz_path in NPZ_PATHS:
    if not os.path.isfile(npz_path):
        print(f"[skip] {npz_path} not found"); continue

    d = np.load(npz_path, allow_pickle=True)
    tr_runs  = d["train_scores_runs"].astype(float)
    te_runs  = d["test_scores_runs"].astype(float)
    tr_files = d["train_files"].astype(object)
    te_files = d["test_files"].astype(object)

    tag = os.path.basename(npz_path).replace(".npz", "")
    mode_name = _check_mode()

    # Ensemble (mean/median/vote) on per-run scores
    tr_ens, te_ens, _ = ensemble_scores(
        tr_runs, te_runs, zscore_per_run=ZSCORE_PER_RUN_BEFORE_ENSEMBLE
    )

    # Optional: restrict TRAIN used for threshold calibration via filename regex
    tr_calib = filter_train_for_calibration(tr_ens, tr_files, CALIBRATION_FILTER_REGEX)

    thresholds = build_thresholds(tr_calib)

    # chronology for TEST files
    ord_idx, days = chronology_indices(te_files)
    inv_ord = np.empty_like(ord_idx)
    inv_ord[ord_idx] = np.arange(len(ord_idx))  # rank of each file in chronological order

     # >>> NEW <<< Natural order for everything below
    nat_idx = natural_order_indices(te_files)           # indices of files in natural order
    inv_nat = np.empty_like(nat_idx); inv_nat[nat_idx] = np.arange(len(nat_idx))


    # --- Master CSV with all thresholds (NATURAL ORDER for order_index)
    thr_names = sorted(list(thresholds.keys()))
    master_csv = os.path.join(OUT_DIR, f"{tag}_{mode_name}_ALL_THRESHOLDS.csv")
    with open(master_csv, "w", newline="") as f:
        w = csv.writer(f)
        header = ["filename", "score", "order_index", "day"] + [f"flagged_{t}" for t in thr_names]
        w.writerow(header)
        for i in range(len(te_files)):
            row = [os.path.basename(str(te_files[i])),
                   float(te_ens[i]),
                   int(inv_nat[i]),   # >>> NEW <<< natural-order rank
                   ""                 # keep column for compatibility; no Day() used
            ]
            for t in thr_names:
                tau = thresholds[t]
                if LOWER_TAIL:
                    fl = (te_ens[i] <= (tau + EPS))      # keep symmetry if LOWER_TAIL ever True
                else:
                    fl = (te_ens[i] >= (tau - EPS))      # >>> NEW <<< EPS + >=
                row.append(int(fl))
            w.writerow(row)
    print(f"[saved] {master_csv}")

    # --- Per-threshold CSVs, counts, earliest+after (ALL in NATURAL ORDER)
    counts_csv = os.path.join(OUT_DIR, f"{tag}_{mode_name}_threshold_counts.csv")
    earliest_csv = os.path.join(OUT_DIR, f"{tag}_{mode_name}_earliest_and_after.csv")
    with open(counts_csv, "w", newline="") as fc, open(earliest_csv, "w", newline="") as fe:
        wc = csv.writer(fc); we = csv.writer(fe)
        wc.writerow(["threshold_name", "tau", "n_anomalous", "n_normal"])
        we.writerow(["threshold_name", "tau",
                     "earliest_index", "earliest_day", "earliest_filename",
                     "n_anom_after_inclusive", "n_norm_after_inclusive"])

        for t in thr_names:
            tau = thresholds[t]
            if LOWER_TAIL:
                flags = (te_ens <= (tau + EPS))
            else:
                flags = (te_ens >= (tau - EPS))  # >>> NEW <<< EPS + >=

            # Per-threshold file (rows in ORIGINAL list order; index column is NATURAL rank)
            thr_csv = os.path.join(OUT_DIR, f"{tag}_{mode_name}_{t}.csv")
            with open(thr_csv, "w", newline="") as ft:
                wt = csv.writer(ft)
                wt.writerow(["filename", "score", "flagged", "order_index", "day"])
                for i in range(len(te_files)):
                    wt.writerow([
                        os.path.basename(str(te_files[i])),
                        float(te_ens[i]),
                        int(flags[i]),
                        int(inv_nat[i]),   # >>> NEW <<< natural-order rank
                        ""                 # no Day()
                    ])
            print(f"[saved] {thr_csv}")

            # Totals (order independent)
            n_anom = int(flags.sum())
            n_norm = int((~flags).sum())
            wc.writerow([t, f"{tau:.6f}", n_anom, n_norm])
            print(f"[{t}] τ={tau:.6f} | anomalies={n_anom} | normals={n_norm}")

            # >>> NEW <<< Earliest + after by NATURAL ORDER ONLY
            if n_anom == 0:
                we.writerow([t, f"{tau:.6f}", "", "", "", 0, len(flags)])
                continue

            flags_nat = flags[nat_idx]                     # reorder flags into natural order
            first_hit_idx = int(np.argmax(flags_nat))      # first True in natural order
            if flags_nat[first_hit_idx] == 0:
                we.writerow([t, f"{tau:.6f}", "", "", "", 0, len(flags)])
                continue

            file_idx = nat_idx[first_hit_idx]
            earliest_file = os.path.basename(str(te_files[file_idx]))
            anom_after = int(flags_nat[first_hit_idx:].sum())
            norm_after = int((~flags_nat[first_hit_idx:]).sum())

            we.writerow([t, f"{tau:.6f}",
                         first_hit_idx,   # natural-order index
                         "",              # no day
                         earliest_file,
                         anom_after,
                         norm_after])

    print(f"[saved] {counts_csv}")
    print(f"[saved] {earliest_csv}")

    # >>> NEW <<< Console printout matching AIRL style (NATURAL ORDER + EPS-safe >=)
print("\n--- Earliest anomaly by threshold (natural order) ---")
for t in thr_names:
    tau = thresholds[t]
    if LOWER_TAIL:
        flags = (te_ens <= (tau + EPS))
    else:
        flags = (te_ens >= (tau - EPS))

    n_anom = int(flags.sum())
    n_norm = int((~flags).sum())
    print(f"Threshold ({t}): {tau:.6f} | Anomalous: {n_anom} | Normal: {n_norm}")

    if n_anom == 0:
        print("  No anomalies detected.")
        continue

    # natural-order earliest
    flags_nat = flags[nat_idx]
    first_hit_idx = int(np.argmax(flags_nat))  # first True in natural order
    if flags_nat[first_hit_idx] == 0:
        print("  No anomalies detected.")
        continue

    file_idx = nat_idx[first_hit_idx]
    earliest_file = os.path.basename(str(te_files[file_idx]))
    print(f"  Earliest Anomalous File: {earliest_file}")
